In [ ]:
import json
import math
from pathlib import Path
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
DOCS_DIR        = "../../data/parsed_json/"
CHROMA_DIR      = "../../data/chroma_db/"
COLLECTION_NAME = "rfp_docs"
MAX_CHUNK_SIZE  = 1000
BATCH_SIZE      = 500

# 임베딩 모델 선택
# "text-embedding-3-small" : 베이스라인, OpenAI API
# "text-embedding-3-large" : 고성능, OpenAI API
# "bge-m3"                 : 한국어 강점, 로컬 무료
# "ko-sroberta-multitask"  : 한국어 특화, 로컬 무료
# "embed-multilingual-v3"  : Cohere, 한국어 포함 100개 언어
EMBEDDING_MODEL = "bge-m3"

1. 유틸리티

In [ ]:
def clean(val):
    # NaN / None → 빈 문자열 (Chroma 메타데이터는 NaN·None 불가)
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [ ]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    # 청크 메타데이터(페이로드) 생성
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }

In [ ]:
#  Chunk dataclass
from dataclasses import dataclass, field

@dataclass
class Chunk:
    content: str
    doc_id: str
    block_id: str
    header_path: str
    metadata: dict = field(default_factory=dict)

2. 청킹

In [ ]:
def chunk_section(section: dict) -> list[dict]:

    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue
        if block["type"] == "table":
            # 쌓인 텍스트 먼저 방출
            if current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = ""
                last_text_block = None
            # 표 단독 방출
            results.append({
                "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                "block":   block,
            })
        else:
            # 텍스트 블록 누적
            if len(current_text) + len(block["content"]) > MAX_CHUNK_SIZE and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    # 남은 텍스트 방출
    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })

    return results

In [ ]:
def process_doc(doc: dict) -> tuple[list[str], list[dict]]:
    # JSON 문서 1개 → (청크 텍스트 리스트, 메타데이터 리스트)
    texts, metas = [], []

    # extraction_warnings 있으면 경고 출력 후 계속 처리
    warnings = doc.get("qa", {}).get("extraction_warnings", [])
    if warnings:
        print(f"  [WARN] {doc['doc_id']} — extraction_warnings: {warnings}")

    meta = doc.get("metadata", {})
    summary = str(clean(meta.get("사업요약")))
    사업명 = str(clean(meta.get("사업명")))
    발주기관 = str(clean(meta.get("발주기관")))

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            # 사업명·발주기관·요약을 앞에 붙여 retrieval 정확도 향상
            prefix = (
                f"[사업명] {사업명}\n"
                f"[발주기관] {발주기관}\n"
                f"[요약] {summary}\n\n"
            )
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

3. 임베딩 모델

In [ ]:
def load_embedding_model(name: str):
    if name == "text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif name == "text-embedding-3-large":
        return OpenAIEmbeddings(model="text-embedding-3-large")
    elif name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "ko-sroberta-multitask":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "embed-multilingual-v3":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model="embed-multilingual-v3.0")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")


4. Vector DB / Retrieval

In [ ]:
def get_vectorstore(embedding_model=None) -> Chroma:
    # 저장된 Chroma DB 불러오기 (검색 전용)
    if embedding_model is None:
        embedding_model = load_embedding_model(EMBEDDING_MODEL)
    return Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=CHROMA_DIR,
    )

In [ ]:
def get_retriever(vectorstore: Chroma, search_type: str = "similarity", k: int = 5):

    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k, "fetch_k": k * 4, "lambda_mult": 0.5},
        )
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

In [ ]:
def save_vectorstore():
    docs_dir = Path(DOCS_DIR)
    json_files = sorted(docs_dir.glob("*.json"))

    all_texts, all_metas = [], []

    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)

        texts, metas = process_doc(doc)
        all_texts.extend(texts)
        all_metas.extend(metas)

    embedding_model = load_embedding_model(EMBEDDING_MODEL)
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=CHROMA_DIR,
    )

    for start in range(0, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vectorstore.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"\nChroma 저장 완료 (총 {vectorstore._collection.count()}개 청크)")

5. 실행

In [ ]:
save_vectorstore()

6. 검색 테스트

In [ ]:
vs = get_vectorstore()
retriever = get_retriever(vs, k=5)

query = "보안 요구사항이 있는 사업은?"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"[{i}] 사업명: {doc.metadata.get('사업명')}")
    print(f"     발주기관: {doc.metadata.get('발주기관')}")
    print(f"     섹션: {doc.metadata.get('header_path')}")
    print(f"     내용:\n{doc.page_content[:300]}")
    print()

7. 임베딩 모델 비교

8. 청크 크기 실험

In [ ]:
def save_vectorstore_with_size(chunk_size: int):
    global MAX_CHUNK_SIZE
    MAX_CHUNK_SIZE = chunk_size

    docs_dir = Path(DOCS_DIR)
    json_files = sorted(docs_dir.glob("*.json"))
    all_texts, all_metas = [], []

    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        texts, metas = process_doc(doc)
        all_texts.extend(texts)
        all_metas.extend(metas)

    chroma_dir = f"../../data/chroma_db_{chunk_size}"
    embedding_model = load_embedding_model(EMBEDDING_MODEL)
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=chroma_dir,
    )
    for start in range(0, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vectorstore.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )

    avg_len = sum(len(t) for t in all_texts) / len(all_texts)
    print(f"chunk_size={chunk_size} | 총 청크: {len(all_texts)} | 평균 길이: {avg_len:.0f}자")
    return vectorstore, all_texts

In [ ]:
# 골든셋 없이 질의 기반으로 눈으로 확인
CHUNK_SIZES = [300, 500, 1000, 2000]
TEST_QUERIES = [
    "보안 요구사항이 있는 사업은?",
    "사업 예산이 가장 큰 사업은?",
    "클라우드 전환 관련 사업은?",
]

for size in CHUNK_SIZES:
    print(f"\n{'='*50}")
    print(f"청크 크기: {size}자")
    print(f"{'='*50}")
    vs, texts = save_vectorstore_with_size(size)
    retriever = get_retriever(vs, k=3)

    for query in TEST_QUERIES:
        print(f"\n[질의] {query}")
        results = retriever.invoke(query)
        for i, doc in enumerate(results, 1):
            print(f"  [{i}] {doc.metadata.get('사업명')} | {doc.metadata.get('header_path')}")

## 청크 크기 실험 결과

| 청크 크기 | 총 청크 수 | 평균 길이(자) | 검색 품질 (주관) | 비고 |
|----------|-----------|-------------|----------------|------|
| 300  | - | - | - | 맥락 부족 우려 |
| 500  | - | - | - | |
| 1000 | 17382 | - | - | 현재 베이스라인 |
| 2000 | - | - | - | 노이즈 우려 |

**채택 크기**: (실험 후 작성)  
**선택 근거**: (실험 후 작성)